# Scaffold plank counting: YOLOv8n with leave-one-image-out
Runtime: **Runtime > Change runtime type > T4 GPU**. The images are confidential: use them only in this notebook and delete the runtime files afterwards.

Steps: (1) upload `scaffold_data.zip` with your **corrected** labels, (2) run the leave-one-out evaluation, (3) train on all 5 and export Core ML.

In [ ]:
!pip -q install ultralytics coremltools
from google.colab import files
up = files.upload()            # choose scaffold_data.zip (with corrected labels)
!unzip -oq scaffold_data.zip
!ls scaffold_data scaffold_data/images

In [ ]:
import os, shutil, time, glob, cv2, numpy as np, torch
from ultralytics import YOLO

DATA = 'scaffold_data'
NAMES = ['img1','img2','img3','img4','img5']
# Correct counts for the FRONT stacks only (image 1 and 4 are approximate)
TRUTH = {'img1':57,'img2':23,'img3':72,'img4':15,'img5':60}
# Front-stack regions (x0, x1, y0, y1); only detections whose centre falls inside are counted
ROIS = {
 'img1': [(0,85,230,447),(90,185,225,447),(188,285,215,447),(290,400,205,447)],
 'img2': [(5,120,235,447),(125,250,250,447),(240,350,285,447)],
 'img3': [(60,135,60,372),(140,218,60,372),(222,295,60,372),(300,378,60,372)],
 'img4': [(0,200,515,700),(205,480,510,700),(485,700,520,700)],
 'img5': [(10,150,25,740),(160,310,15,770),(325,480,5,785)],
}
REPEAT = 16     # each training image is repeated with different names so one epoch has enough steps

In [ ]:
def build_split(train_names, out):
    shutil.rmtree(out, ignore_errors=True)
    for sub in ['images/train','labels/train','images/val','labels/val']:
        os.makedirs(f'{out}/{sub}')
    for n in train_names:
        for r in range(REPEAT):
            shutil.copy(f'{DATA}/images/{n}.png', f'{out}/images/train/{n}_{r}.png')
            shutil.copy(f'{DATA}/labels/{n}.txt', f'{out}/labels/train/{n}_{r}.txt')
    v = train_names[0]            # placeholder val (val is switched off during training)
    shutil.copy(f'{DATA}/images/{v}.png', f'{out}/images/val/{v}.png')
    shutil.copy(f'{DATA}/labels/{v}.txt', f'{out}/labels/val/{v}.txt')
    open(f'{out}/data.yaml','w').write(f'path: {os.path.abspath(out)}\ntrain: images/train\nval: images/val\nnames: {{0: plank}}\n')
    return f'{out}/data.yaml'

def train(yaml, name, epochs=60):
    m = YOLO('yolov8n.pt')
    m.train(data=yaml, epochs=epochs, imgsz=800, batch=16, val=False, plots=False, verbose=False,
            mosaic=1.0, degrees=5, scale=0.5, translate=0.1, perspective=0.0005,
            hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, fliplr=0.5, flipud=0.0,
            project='runs', name=name, exist_ok=True)
    return YOLO(f'runs/{name}/weights/best.pt')

def count_front(model, name, conf=0.25):
    img = cv2.imread(f'{DATA}/images/{name}.png')
    t = time.perf_counter(); res = model.predict(img, imgsz=800, conf=conf, iou=0.5, verbose=False)[0]
    ms = (time.perf_counter()-t)*1000
    boxes = res.boxes.xyxy.cpu().numpy(); keep=[]
    for b in boxes:
        cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
        if any(x0<=cx<=x1 and y0<=cy<=y1 for x0,x1,y0,y1 in ROIS[name]): keep.append(b)
    vis = img.copy()
    for b in keep: cv2.rectangle(vis,(int(b[0]),int(b[1])),(int(b[2]),int(b[3])),(0,0,255),1)
    os.makedirs('results', exist_ok=True); cv2.imwrite(f'results/{name}_pred.png', vis)
    return len(keep), len(boxes), ms

In [ ]:
rows = []
for held in NAMES:
    yaml = build_split([n for n in NAMES if n != held], f'split_{held}')
    model = train(yaml, f'loo_{held}')
    count_front(model, held)                       # warm-up
    n, n_all, ms = np.mean([count_front(model, held)[0] for _ in range(1)]), None, None
    n, n_all, ms = count_front(model, held)
    rows.append((held, TRUTH[held], n, n - TRUTH[held], round(ms,1)))
    print(rows[-1])

print('\nimage | correct | predicted (front stacks) | error | inference ms (' + ('GPU' if torch.cuda.is_available() else 'CPU') + ')')
for r in rows: print(*r, sep=' | ')
print('mean absolute error:', np.mean([abs(r[3]) for r in rows]))

## Read the results honestly
Each image is predicted by a model that **never saw that image** (trained on the other four). With so little data expect misses on images 1, 2 and 4. Look at `results/*_pred.png`, and note the error cases (occlusion, cropped bottoms, long plank tops) for the report. Try `conf` 0.15 to 0.4 on one image only to show sensitivity; do not tune it per image.

In [ ]:
# Final model: train on all 5 images, export for iOS
yaml = build_split(NAMES, 'split_all')
final = train(yaml, 'final', epochs=80)
final.export(format='coreml', imgsz=800, nms=True)    # produces runs/final/weights/best.mlpackage
!ls runs/final/weights

In [ ]:
# CPU inference time (closer to a phone than a T4); run on a CPU runtime or force device='cpu'
m = YOLO('runs/final/weights/best.pt'); img = cv2.imread(f'{DATA}/images/img3.png')
m.predict(img, imgsz=800, device='cpu', verbose=False)
ts=[]
for _ in range(10):
    t=time.perf_counter(); m.predict(img, imgsz=800, device='cpu', verbose=False); ts.append((time.perf_counter()-t)*1000)
print('CPU ms per image (median of 10):', round(float(np.median(ts)),1))
!zip -qr outputs.zip results runs/final/weights
from google.colab import files; files.download('outputs.zip')